<a href="https://colab.research.google.com/github/kauesmatheus/Agente-de-RH/blob/main/AGENTE_RH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-groq \
langchain-huggingface \
chromadb \
pypdf \
groq \
pyngrok \
streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 

In [7]:
import os
from groq import Groq
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [11]:
%%writefile app.py

import os
import streamlit as st
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq


# CONFIG
PERSIST_DIRECTORY = './chroma_rh'

st.set_page_config(page_title="Chatbot RH", layout="wide")


# ESTADO GLOBAL
if "vectorstore" not in st.session_state:
    st.session_state.vectorstore = None


# FUNÇÕES
@st.cache_data
def carregar_documentos():
    caminhos = [
        "politica_ferias.pdf",
        "politica_home_office.pdf",
        "codigo_conduta.pdf"
    ]

    documentos = []
    for caminho in caminhos:
        if not os.path.exists(caminho):
            st.error(f"Arquivo não encontrado: {caminho}")
            continue

        loader = PyPDFLoader(caminho)
        docs = loader.load()

        for doc in docs:
            doc.metadata["documento"] = caminho

        documentos.extend(docs)

    return documentos


def gerar_chunks(documentos):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )
    return splitter.split_documents(documentos)


def enriquecer_chunks(chunks):
    for chunk in chunks:
        texto = chunk.page_content.lower()

        if "ferias" in texto:
            chunk.metadata["categoria"] = "ferias"
        elif "home office" in texto or "remoto" in texto:
            chunk.metadata["categoria"] = "home_office"
        elif "conduta" in texto or "etica" in texto:
            chunk.metadata["categoria"] = "conduta"
        else:
            chunk.metadata["categoria"] = "geral"

    return chunks


@st.cache_resource
def criar_vectorstore(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=PERSIST_DIRECTORY
    )

    return vectorstore


def rerank_documentos(pergunta, documentos, llm):
    prompt_rerank = PromptTemplate(
        input_variables=["pergunta", "texto"],
        template="""
Dê uma nota de 0 a 10 para o quão relevante o texto abaixo é para responder a pergunta.

Pergunta: {pergunta}
Texto: {texto}

Responda apenas com um número.
"""
    )

    documentos_com_score = []
    for doc in documentos:
        score = llm.invoke(
            prompt_rerank.format(
                pergunta=pergunta,
                texto=doc.page_content
            )
        ).content

        try:
            score = float(score)
        except:
            score = 0

        documentos_com_score.append((score, doc))

    # Ordena do mais relevante para o menos relevante
    documentos_ordenados = sorted(
        documentos_com_score,
        key=lambda x: x[0],
        reverse=True
    )
    # Retorna apenas os documentos
    return [doc for _, doc in documentos_ordenados]


def responder_pergunta(pergunta, vectorstore):

    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )

    # Recuperação
    documentos_recuperados = vectorstore.similarity_search(
        pergunta,
        k=8
    )

    # Reranking
    documentos_rerankeados = rerank_documentos(
        pergunta,
        documentos_recuperados,
        llm
    )

    # Seleciona os melhores
    contexto_final = documentos_rerankeados[:4]

    contexto_texto = "\n\n".join(
        [doc.page_content for doc in contexto_final]
    )

    prompt_final = f"""
Responda a pergunta com base no contexto abaixo:

{contexto_texto}

Pergunta: {pergunta}
"""

    resposta = llm.invoke(prompt_final)

    # AGORA RETORNAMOS TAMBÉM OS CHUNKS
    return resposta.content, contexto_final


# INTERFACE
st.title("Agente de RH")
st.write("Consulta inteligente sobre relações humanas.")

pergunta = st.text_input("Digite sua pergunta:")


# BOTÃO PROCESSAR
if st.button("Processar documentos"):

    try:
        with st.spinner("Carregando documentos..."):
            docs = carregar_documentos()

        if not docs:
            st.error("Nenhum documento carregado.")
        else:
            chunks = gerar_chunks(docs)
            chunks = enriquecer_chunks(chunks)

            with st.spinner("Criando base vetorial..."):
                vectorstore = criar_vectorstore(chunks)

            st.session_state.vectorstore = vectorstore

            st.success("Base criada com sucesso!")

    except Exception as e:
        st.error("Erro ao processar documentos:")
        st.text(str(e))


# PERGUNTA
if pergunta:

    if st.session_state.vectorstore is None:
        st.warning(" Clique primeiro em 'Processar documentos'")
    else:
        try:
            with st.spinner("Gerando resposta..."):
                resposta, fontes = responder_pergunta(
                pergunta,
                st.session_state.vectorstore
            )

            st.write("### Resposta:")
            st.write(resposta)

            # EXIBIR FONTES
            st.write("### Fontes utilizadas:")

            for i, doc in enumerate(fontes):
                with st.expander(f"Chunk {i+1}"):
                    st.write("Documento:", doc.metadata.get("documento", "N/A"))
                    st.write("Categoria:", doc.metadata.get("categoria", "N/A"))
                    st.write("Conteúdo:")
                    st.write(doc.page_content[:500])  # limita tamanho

        except Exception as e:
            st.error("Erro ao gerar resposta:")
            st.text(str(e))

Overwriting app.py


In [13]:
# VERIFICAR APP
import os

print("\nVerificando arquivos no diretório:")
print(os.listdir())

if not os.path.exists("app.py"):
    print("ERRO: app.py NÃO encontrado!")
else:
    print("app.py encontrado!")


# NGROK
from pyngrok import ngrok

print("\n Iniciando ngrok...")

try:
    ngrok.set_auth_token("3B8AiMz7LHgMKyRqKATM04FEJOr_6KQUpa28WRCosBfafeK1w")

    public_url = ngrok.connect(8501)
    print("NGROK OK!")
    print("LINK:", public_url)

except Exception as e:
    print("ERRO no ngrok:", e)


# STREAMLIT
print("\n Iniciando Streamlit...")

try:
    get_ipython().system_raw("streamlit run app.py --server.port 8501 &")
    print("Streamlit iniciado na porta 8501")

except Exception as e:
    print("ERRO no Streamlit:", e)


# MONITORAMENTO
import time

print("\n Aguardando inicialização...")
time.sleep(5)

print("\n Verificando portas abertas:")
!lsof -i -P -n | grep LISTEN

print("\n FINALIZADO")
print("Se o link não abriu, o erro está acima")


Verificando arquivos no diretório:
['.config', 'politica_ferias.pdf', 'codigo_conduta.pdf', 'app.py', 'politica_home_office.pdf', 'sample_data']
app.py encontrado!

 Iniciando ngrok...
NGROK OK!
LINK: NgrokTunnel: "https://historiographical-karis-pasquillic.ngrok-free.dev" -> "http://localhost:8501"

 Iniciando Streamlit...
Streamlit iniciado na porta 8501

 Aguardando inicialização...



 Verificando portas abertas:
node         7 root   21u  IPv6  19634      0t0  TCP *:8080 (LISTEN)
kernel_ma   25 root    4u  IPv4  18979      0t0  TCP 172.28.0.12:6000 (LISTEN)
colab-fil   71 root    3u  IPv4  19312      0t0  TCP 127.0.0.1:3453 (LISTEN)
jupyter-s   92 root    7u  IPv4  19451      0t0  TCP 172.28.0.12:9000 (LISTEN)
python3    356 root   28u  IPv4  26139      0t0  TCP 127.0.0.1:40933 (LISTEN)
ngrok     2451 root    3u  IPv4  77037      0t0  TCP 127.0.0.1:4040 (LISTEN)
streamlit 2466 root   13u  IPv4  76580      0t0  TCP *:8501 (LISTEN)
streamlit 2466 root   14u  IPv6  76581      0t0  TCP *:8501 (LISTEN)

 FINALIZADO
Se o link não abriu, o erro está acima


In [12]:
# LIMPEZA
print("Limpando processos antigos...")
!killall streamlit > /dev/null 2>&1
!killall ngrok > /dev/null 2>&1

Limpando processos antigos...
